# Exemplo comparativo para o artigo

Produz o material visual que ilustra a diferença entre um relatório em que a seção contábil foi
localizada e outro em que não foi. A saída vai para `exemplo/` no Drive: duas imagens PNG, um
resumo em texto e uma tabela com o diagnóstico dos dois documentos.

**O que a comparação mostra.** Para cada documento, o programa registra quais dos seis sinais
contábeis aparecem no melhor trecho candidato, quantos valores numéricos ele contém e quais
âncoras foram encontradas — o título do Item 8, o parecer do auditor, o título do balanço. É a
diferença entre essas duas fichas que explica, em termos objetivos, por que um foi extraído e o
outro não.

**Ordem.** Execute de cima para baixo. A seção 3 lista os casos disponíveis e você escolhe o par;
a seção 4 produz as imagens e o resumo.

## 1. Ambiente

In [ ]:
SEU_NOME  = "Helena Ribeiro"
SEU_EMAIL = "helenafariasr@gmail.com"

USAR_DRIVE = True
PASTA      = "/content/drive/MyDrive/SEC_PAINEL"

FORMULARIOS = ["10-K", "20-F", "40-F"]
SALVAR_HTML_DA_SECAO = False

REQ_POR_SEGUNDO = 8
N_THREADS       = 4
MAX_TENTATIVAS  = 4

In [ ]:
import os

if USAR_DRIVE:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive", force_remount=True)
        print("Drive montado.")
    except Exception as e:
        print("Não foi possível montar o Drive:", type(e).__name__, e)
        USAR_DRIVE = False
        PASTA = PASTA.replace("/content/drive/MyDrive", "/content")

for sub in ["", "indice", "pdf", "html_secao", "log"]:
    os.makedirs(os.path.join(PASTA, sub), exist_ok=True)
print("Pasta de trabalho:", PASTA)

!apt-get -qq update > /dev/null && apt-get -qq install -y wkhtmltopdf poppler-utils > /dev/null
!wkhtmltopdf --version | head -1 && pdftotext -v 2>&1 | head -1

In [ ]:
import io, os, re, json, time, random, threading, subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urljoin

import requests
import pandas as pd

assert "@" in SEU_EMAIL and SEU_NOME.strip(), "Preencha SEU_NOME e SEU_EMAIL."

HEADERS = {"User-Agent": f"{SEU_NOME} {SEU_EMAIL}", "Accept-Encoding": "gzip, deflate"}

_lock, _ultimo = threading.Lock(), [0.0]

def _esperar_vez():
    with _lock:
        agora = time.time()
        espera = _ultimo[0] + 1.0 / REQ_POR_SEGUNDO - agora
        if espera > 0:
            time.sleep(espera)
            agora = time.time()
        _ultimo[0] = agora

_local = threading.local()

def _sessao():
    if not hasattr(_local, "s"):
        s = requests.Session(); s.headers.update(HEADERS); _local.s = s
    return _local.s

def baixar(url, binario=False):
    """GET com limite de taxa, repetição e espera crescente. None se falhar."""
    for i in range(MAX_TENTATIVAS):
        _esperar_vez()
        try:
            r = _sessao().get(url, timeout=90)
            if r.status_code == 200:
                return r.content if binario else r.text
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep((2 ** i) + random.random())
    return None

print("Identificação enviada à SEC:", HEADERS["User-Agent"])

## 2. Motor de recorte

In [ ]:
IGNORAR = re.compile(r"(?i)^(r\d+\.htm|report\d*\.htm|.*-index.*\.html?|"
                     r"\d{10}-\d{2}-\d{6}\.txt|.*_?cal\.|.*_?def\.|.*_?lab\.|.*_?pre\.|"
                     r".*\.xml|.*\.xsd|.*\.jpg|.*\.png|.*\.gif)$")
EXTENSOES = (".htm", ".html", ".txt", ".pdf")

def arquivos_do_protocolo(url_pasta, excluir=()):
    """Documentos do protocolo, do maior para o menor, incluindo os anexos em PDF."""
    txt = baixar(urljoin(url_pasta, "index.json"))
    if not txt:
        return []
    try:
        itens = json.loads(txt)["directory"]["item"]
    except (ValueError, KeyError):
        return []
    saida = []
    for i in itens:
        nome = i.get("name", "")
        if not nome.lower().endswith(EXTENSOES):
            continue
        if IGNORAR.match(nome) or nome in excluir:
            continue
        saida.append((nome, int(i.get("size") or 0)))
    saida.sort(key=lambda x: x[1], reverse=True)
    return [n for n, _ in saida]

P_DOCUMENTO = re.compile(r"(?is)<DOCUMENT>.*?<TYPE>([^\r\n<]*).*?<FILENAME>([^\r\n<]*)"
                         r".*?<TEXT>(.*?)</TEXT>")

def documentos_da_submissao(url_submissao):
    """Último recurso: a submissão completa reúne todos os documentos do protocolo."""
    bruto = baixar(url_submissao)
    if not bruto:
        return []
    saida = []
    for tipo, nome, corpo in P_DOCUMENTO.findall(bruto):
        nome = nome.strip()
        if not nome.lower().endswith((".htm", ".html", ".txt")):
            continue
        if len(corpo) > 40000:
            saida.append((nome, corpo))
    saida.sort(key=lambda x: len(x[1]), reverse=True)
    return saida[:6]

def documento_principal(linha):
    """O documento principal vem da tabela do painel; a listagem da pasta é a alternativa."""
    doc = linha.get("documento_principal")
    if not (isinstance(doc, str) and doc):
        nomes = arquivos_do_protocolo(linha["url_pasta"])
        doc = nomes[0] if nomes else None
    sic = str(linha.get("sic") or "")
    return (linha["url_pasta"] + doc if doc else None), doc, sic

In [ ]:
ENTIDADES = re.compile(r"&nbsp;|&#160;|&#xa0;|&#xA0;")
TAGS      = re.compile(r"(?is)<(script|style)\b.*?</\1>|<[^>]+>")

def texto_e_mapa(html):
    """Texto visível em minúsculas, espaços normalizados, com mapa texto -> posição no HTML."""
    saida, mapa, ultimo, espaco = [], [], 0, True
    def empurrar(ch, pos):
        nonlocal espaco
        if ch.isspace():
            if espaco:
                return
            saida.append(" "); mapa.append(pos); espaco = True
        else:
            saida.append(ch.lower()); mapa.append(pos); espaco = False
    for m in TAGS.finditer(html):
        for k, ch in enumerate(html[ultimo:m.start()]):
            empurrar(ch, ultimo + k)
        empurrar(" ", m.start())
        ultimo = m.end()
    for k, ch in enumerate(html[ultimo:]):
        empurrar(ch, ultimo + k)
    return "".join(saida), mapa

# Títulos de início e de fim
P_10K_INI = re.compile(r"item\s*8[\.\:\)\-—\s]{0,6}financial\s+statements")
P_10K_FIM = re.compile(r"item\s*9[a-c]?[\.\:\)\-—\s]{0,6}(changes\s+in\s+and|controls\s+and\s+procedures|other\s+information)")
P_20F_INI = re.compile(r"item\s*18[\.\:\)\-—\s]{0,6}financial\s+statements")
P_20F_ALT = re.compile(r"item\s*17[\.\:\)\-—\s]{0,6}financial\s+statements")
P_20F_FIM = re.compile(r"item\s*19[\.\:\)\-—\s]{0,6}exhibit")
P_AUDITOR = re.compile(r"report\s+of\s+independent|independent\s+auditor'?s?\s+report|"
                       r"auditors?'?\s+report\s+to|report\s+of\s+the\s+independent")
P_ASSINAT = re.compile(r"\bsignatures?\b")

# ---------------------------------------------------------------------------
# Sinais que caracterizam um conjunto de demonstrações contábeis
# ---------------------------------------------------------------------------
SINAIS = {
    "balanco":    re.compile(r"balance\s+sheets?|statements?\s+of\s+financial\s+position"),
    "notas":      re.compile(r"notes\s+to\s+.{0,60}?financial\s+statements|"
                             r"notes\s+to\s+the\s+accounts"),
    "resultado":  re.compile(r"statements?\s+of\s+((net|total|consolidated|combined)\s+){0,2}"
                             r"(operations|income|comprehensive|profit|earnings|loss)|"
                             r"income\s+statements?"),
    "fluxo":      re.compile(r"statements?\s+of\s+cash\s+flows?|cash\s+flow\s+statements?"),
    "patrimonio": re.compile(r"statements?\s+of\s+(changes\s+in\s+)?"
                             r"(shareholders|stockholders|owners|equity)"),
    "auditor":    P_AUDITOR,
}
OBRIGATORIOS = ("balanco", "notas")     # sem estes dois, o trecho é texto narrativo, não demonstração
PONTOS_MINIMOS = 4                      # de 6 sinais
PONTOS_BONS    = 5                      # a partir daqui, não vale procurar em outros documentos

# Conteúdo numérico: demonstrações contábeis são densas em números; índices e sumários não.
P_NUMERO = re.compile(r"\d{1,3}(?:,\d{3})+|\d+\.\d{2}\b")
MIN_NUMEROS = 150          # abaixo disso o trecho é índice ou remissão, não demonstração
NUMEROS_BONS = 400         # a partir daqui, não vale procurar em outros documentos
RETENCAO_MINIMA = 0.6      # trecho mais curto que preserve ao menos esta fração dos números

# Declarações de ausência (fundos de securitização, empresas-veículo)
P_OMITIDO = re.compile(r"^.{0,400}?(omitted|not\s+applicable|none\.)", re.S)

MIN_ACEITAVEL = 4000

def pontuar(trecho):
    """(sinais contábeis presentes, quantidade de números). Zero sinais se faltar um obrigatório."""
    presentes = {k for k, p in SINAIS.items() if p.search(trecho)}
    numeros = len(P_NUMERO.findall(trecho))
    if any(o not in presentes for o in OBRIGATORIOS):
        return 0, numeros
    return len(presentes), numeros

def melhor_par(texto, p_ini, p_fim):
    """Par (início, fim) mais distante — evita o sumário, onde os títulos ficam colados."""
    inis = [m.start() for m in p_ini.finditer(texto)]
    fins = [m.start() for m in p_fim.finditer(texto)]
    melhor, tamanho = None, 0
    for i in inis:
        seguintes = [f for f in fins if f > i]
        f = seguintes[0] if seguintes else len(texto)
        if f - i > tamanho:
            melhor, tamanho = (i, f), f - i
    return melhor

def candidatos(texto, formulario):
    """Trechos a testar: pelo item, pelo parecer do auditor e, quando cabe, o documento inteiro."""
    saida = []
    if formulario == "10-K":
        p = melhor_par(texto, P_10K_INI, P_10K_FIM)
        if p:
            saida.append((p[0], p[1], "item8"))
    elif formulario == "20-F":
        for padrao, nome in ((P_20F_INI, "item18"), (P_20F_ALT, "item17")):
            p = melhor_par(texto, padrao, P_20F_FIM)
            if p:
                saida.append((p[0], p[1], nome))
    # Âncoras: o parecer do auditor e o título do balanço. As duas são necessárias porque a
    # ordem varia — em boa parte dos arquivamentos europeus o parecer vem depois das
    # demonstrações, e ancorar só nele deixaria as demonstrações de fora do trecho.
    for padrao, nome in ((P_AUDITOR, "paginas_F"), (SINAIS["balanco"], "balanco")):
        for m in list(padrao.finditer(texto))[:5]:
            ini = max(0, m.start() - 60)       # recua o bastante para não cortar o título ao meio
            fins = [s.start() for s in P_ASSINAT.finditer(texto) if s.start() > ini + MIN_ACEITAVEL]
            saida.append((ini, fins[-1] if fins else len(texto), nome))
    if formulario in ("40-F", "anexo"):
        saida.append((0, len(texto), "documento_integral"))
    return saida

def recortar(html, formulario):
    """Melhor trecho do documento: (html_da_secao, metodo, n_caracteres, pontos, numeros).
    metodo 'sem_demonstracoes' quando o item existe e está declarado como omitido."""
    html = ENTIDADES.sub(" ", html)
    texto, mapa = texto_e_mapa(html)
    if not texto:
        return None, "sem_texto", 0, 0, 0

    validos, omitido = [], False
    for ini, fim, metodo in candidatos(texto, formulario):
        trecho = texto[ini:fim]
        if len(trecho) < MIN_ACEITAVEL:
            if P_OMITIDO.search(trecho):
                omitido = True
            continue
        pontos, numeros = pontuar(trecho)
        if pontos < PONTOS_MINIMOS or numeros < MIN_NUMEROS:
            continue
        validos.append((pontos, numeros, ini, fim, metodo, len(trecho)))

    if not validos:
        return None, ("sem_demonstracoes" if omitido else "nao_localizado"), 0, 0, 0

    # Escolha em duas etapas. Primeiro o maior número de sinais e o maior conteúdo numérico,
    # que é o que separa as demonstrações do índice que as lista. Depois, entre os trechos que
    # preservam a maior parte desse conteúdo, o mais curto — assim o resultado fica nas
    # demonstrações em vez de abarcar o relatório inteiro.
    melhor_pontos = max(v[0] for v in validos)
    fortes = [v for v in validos if v[0] == melhor_pontos]
    teto = max(v[1] for v in fortes)
    proximos = [v for v in fortes if v[1] >= RETENCAO_MINIMA * teto]
    pontos, numeros, ini, fim, metodo, n = min(proximos, key=lambda v: v[5])
    ini_html = mapa[ini]
    fim_html = mapa[fim - 1] if fim - 1 < len(mapa) else len(html)
    return html[ini_html:fim_html], metodo, n, pontos, numeros

MOLDE = """<html><head><meta charset="utf-8"><style>
 @page {{ size: A4; margin: 12mm }}
 body {{ font-family: Georgia, serif; font-size: 9pt; line-height: 1.35 }}
 table {{ border-collapse: collapse; font-size: 7.5pt; width: 100% }}
 td, th {{ padding: 1px 3px; vertical-align: bottom }}
 img {{ display: none }}
 .cabecalho {{ font-size: 8pt; color: #555; border-bottom: 1px solid #999;
               margin-bottom: 8pt; padding-bottom: 4pt }}
</style></head><body>
<div class="cabecalho">{titulo}</div>
{corpo}
</body></html>"""

def texto_de_pdf(bytes_pdf):
    """Texto de um anexo já entregue em PDF, para que ele possa ser avaliado como os demais."""
    with tempfile.NamedTemporaryFile("wb", suffix=".pdf", delete=False) as f:
        f.write(bytes_pdf); tmp = f.name
    try:
        r = subprocess.run(["pdftotext", "-q", tmp, "-"], capture_output=True, text=True,
                           timeout=300)
        return r.stdout
    except Exception:
        return ""
    finally:
        os.unlink(tmp)

def avaliar_pdf(bytes_pdf):
    """(pontos, numeros) de um anexo em PDF."""
    txt = re.sub(r"\s+", " ", texto_de_pdf(bytes_pdf)).lower()
    if len(txt) < MIN_ACEITAVEL:
        return 0, 0
    return pontuar(txt)

def gerar_pdf(html_secao, titulo, destino):
    with tempfile.NamedTemporaryFile("w", suffix=".html", delete=False, encoding="utf-8") as f:
        f.write(MOLDE.format(titulo=titulo, corpo=html_secao)); tmp = f.name
    try:
        subprocess.run(["wkhtmltopdf", "--quiet", "--enable-local-file-access", "--no-images",
                        "--load-error-handling", "ignore", "--load-media-error-handling", "ignore",
                        "--disable-external-links", "--footer-right", "[page]/[topage]",
                        "--footer-font-size", "7", tmp, destino],
                       capture_output=True, timeout=600)
        return os.path.exists(destino) and os.path.getsize(destino) > 1000
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(tmp)

In [ ]:
LOG    = os.path.join(PASTA, "log", "extracao.csv")
FALHAS = os.path.join(PASTA, "log", "falhas.csv")
COLUNAS = ("accession,cik,empresa,ticker,formulario,data,sic,metodo,documento,pontos,numeros,"
           "caracteres,kb_pdf,url")
COL_FALHA = "accession,cik,empresa,formulario,data,sic,motivo,url,documentos_examinados"

_log_lock = threading.Lock()

def registrar(caminho, linha, cabecalho):
    with _log_lock:
        novo = not os.path.exists(caminho)
        with open(caminho, "a", encoding="utf-8") as f:
            if novo:
                f.write(cabecalho + "\n")
            f.write(linha + "\n")

def concluidos():
    feitos = set()
    for c in (LOG, FALHAS):
        if os.path.exists(c):
            feitos |= set(pd.read_csv(c)["accession"].astype(str))
    return feitos

def limpo(s):
    return str(s).replace(",", " ").replace("\n", " ")[:80]

def extrair(linha):
    """Percorre os documentos do protocolo e devolve o trecho de melhor conteúdo contábil.
    (conteudo, metodo, n, pontos, numeros, url, documento, sic, examinados)"""
    url, nome, sic = documento_principal(linha)
    melhor, omitido, examinados = None, False, []

    def considerar(bruto, u, doc, forma):
        """Avalia um documento. bruto em texto (HTML) ou bytes (PDF já pronto)."""
        nonlocal melhor, omitido
        examinados.append(doc)
        if isinstance(bruto, bytes):                     # anexo entregue em PDF
            pontos, numeros = avaliar_pdf(bruto)
            if pontos >= PONTOS_MINIMOS and numeros >= MIN_NUMEROS:
                chave = (pontos, numeros)
                if melhor is None or chave > (melhor[3], melhor[4]):
                    melhor = (bruto, "pdf_original", 0, pontos, numeros, u, doc)
                return pontos, numeros
            return 0, 0
        secao, metodo, n, pontos, numeros = recortar(bruto, forma)
        if metodo == "sem_demonstracoes":
            omitido = True
        if secao:
            chave = (pontos, numeros)
            if melhor is None or chave > (melhor[3], melhor[4]):
                melhor = (secao, metodo, n, pontos, numeros, u, doc)
            return pontos, numeros
        return 0, 0

    def bom(par):
        return par[0] >= PONTOS_BONS and par[1] >= NUMEROS_BONS

    if linha["formulario"] != "40-F" and url:
        bruto = baixar(url)
        if bruto and bom(considerar(bruto, url, nome, linha["formulario"])):
            return melhor + (sic, examinados)
        if omitido:                       # item existe e está declarado como omitido
            return (None, "sem_demonstracoes", 0, 0, 0, url, nome, sic, examinados)

    # Demais documentos do protocolo: anexos do 40-F, demonstrações em arquivo separado
    for outro in arquivos_do_protocolo(linha["url_pasta"], excluir=(nome,) if nome else ())[:12]:
        u = linha["url_pasta"] + outro
        binario = outro.lower().endswith(".pdf")
        bruto = baixar(u, binario=binario)
        if not bruto:
            continue
        if bom(considerar(bruto, u, outro, "anexo")):
            break

    # Último recurso: a submissão completa, que traz documentos fora da listagem da pasta
    if melhor is None and not omitido:
        for doc, corpo in documentos_da_submissao(linha["url_submissao_completa"]):
            if bom(considerar(corpo, linha["url_submissao_completa"], doc, "anexo")):
                break

    if melhor:
        conteudo, metodo, n, pontos, numeros, u, doc = melhor
        if doc != nome:
            metodo += "_em_anexo"
        return (conteudo, metodo, n, pontos, numeros, u, doc, sic, examinados)
    return (None, "sem_demonstracoes" if omitido else "nao_localizado",
            0, 0, 0, url, nome, sic, examinados)

def processar(linha):
    acc = linha["accession"]
    try:
        secao, metodo, n, pontos, numeros, url, doc, sic, examinados = extrair(linha)
        base_falha = (f"{acc},{linha['cik']},{limpo(linha['empresa'])},{linha['formulario']},"
                      f"{linha['data']},{sic}")
        if not secao:
            registrar(FALHAS, f"{base_falha},{metodo},{url},{' '.join(examinados[:10])}",
                      COL_FALHA)
            return metodo if metodo == "sem_demonstracoes" else "falha"

        pasta_ano = os.path.join(PASTA, "pdf", str(linha["ano"]), linha["formulario"])
        os.makedirs(pasta_ano, exist_ok=True)
        tic = linha.get("ticker") if pd.notna(linha.get("ticker")) else "NA"
        base = f"{linha['cik']}_{tic}_{linha['ano']}_{acc}"
        destino = os.path.join(pasta_ano, base + ".pdf")

        titulo = (f"{limpo(linha['empresa'])} — CIK {linha['cik']} — {linha['formulario']} — "
                  f"protocolo {linha['data']} — accession {acc}")
        if isinstance(secao, bytes):          # anexo já entregue em PDF pela própria empresa
            with open(destino, "wb") as f:
                f.write(secao)
        elif not gerar_pdf(secao, titulo, destino):
            registrar(FALHAS, f"{base_falha},pdf_falhou,{url},", COL_FALHA)
            return "falha"

        if SALVAR_HTML_DA_SECAO and not isinstance(secao, bytes):
            ph = os.path.join(PASTA, "html_secao", str(linha["ano"]))
            os.makedirs(ph, exist_ok=True)
            with open(os.path.join(ph, base + ".html"), "w", encoding="utf-8") as f:
                f.write(secao)

        kb = os.path.getsize(destino) // 1024
        registrar(LOG, f"{acc},{linha['cik']},{limpo(linha['empresa'])},{tic},"
                       f"{linha['formulario']},{linha['data']},{sic},{metodo},{doc},{pontos},"
                       f"{numeros},{n},{kb},{url}", COLUNAS)
        return "ok"
    except Exception as e:
        registrar(FALHAS, f"{acc},{linha['cik']},{limpo(linha.get('empresa'))},"
                          f"{linha['formulario']},{linha['data']},,{type(e).__name__},,",
                  COL_FALHA)
        return "erro"

## 3. Escolha do par

A primeira tabela lista as falhas de localização, ordenadas por nome. Escolha uma empresa
reconhecível — uma operacional de porte médio funciona melhor na figura do que um trust obscuro.
A segunda lista candidatos ao contraste: extraídos com sucesso, do mesmo formulário e ano.

In [ ]:
fa = pd.read_csv(FALHAS, low_memory=False)
lg = pd.read_csv(LOG, low_memory=False)
trab = pd.read_csv(os.path.join(PASTA, "indice", "trabalho.csv"), low_memory=False)

perdidos = fa[fa["motivo"] == "nao_localizado"].merge(
    trab[["accession", "ano", "url_pasta", "documento_principal"]],
    on="accession", how="left")

print("falhas de localização:", len(perdidos), "\n")
print(perdidos[["empresa", "formulario", "ano", "sic", "accession"]]
      .sort_values("empresa").head(40).to_string(index=False))

In [ ]:
# Escolha aqui. Troque o accession da falha por um da lista acima, se quiser outro exemplo.
ACC_FALHA = perdidos.sort_values("empresa")["accession"].iloc[0]

linha_falha = trab[trab["accession"] == ACC_FALHA].iloc[0]
print("falha escolhida:", linha_falha["empresa"], linha_falha["ano"], linha_falha["formulario"])

pares = lg[(lg["formulario"] == linha_falha["formulario"])
           & (lg["data"].str[:4] == str(linha_falha["ano"]))
           & (lg["metodo"] == "item8")
           & (lg["numeros"] > 500)]
print("\ncandidatos a contraste:")
print(pares[["empresa", "formulario", "metodo", "pontos", "numeros", "accession"]]
      .head(15).to_string(index=False))

In [ ]:
ACC_OK = pares["accession"].iloc[0]     # ou escolha outro da lista
linha_ok = trab[trab["accession"] == ACC_OK].iloc[0]
print("contraste escolhido:", linha_ok["empresa"], linha_ok["ano"], linha_ok["formulario"])

## 4. Diagnóstico dos dois documentos

Baixa o documento principal de cada um, aplica o mesmo motor de recorte e registra o resultado:
sinais contábeis presentes, conteúdo numérico e âncoras localizadas.

In [ ]:
def ficha(linha):
    """Diagnóstico de um documento: âncoras, sinais e conteúdo numérico."""
    url, doc, sic = documento_principal(linha)
    bruto = baixar(url)
    if not bruto:
        return {"empresa": linha["empresa"], "erro": "documento não baixou"}, None, None

    html = ENTIDADES.sub(" ", bruto)
    texto, _ = texto_e_mapa(html)

    ancoras = {
        "titulo do item": bool(P_10K_INI.search(texto) or P_20F_INI.search(texto)),
        "parecer do auditor": bool(P_AUDITOR.search(texto)),
        "titulo do balanco": bool(SINAIS["balanco"].search(texto)),
    }

    secao, metodo, n, pontos, numeros = recortar(bruto, linha["formulario"])
    presentes = {k: bool(p.search(texto)) for k, p in SINAIS.items()}

    f = {"empresa": linha["empresa"], "formulario": linha["formulario"],
         "ano": linha["ano"], "accession": linha["accession"],
         "documento": doc, "tamanho_do_texto": len(texto),
         "resultado": metodo, "pontos": pontos, "numeros": numeros}
    f.update({"ancora_" + k.replace(" ", "_"): v for k, v in ancoras.items()})
    f.update({"sinal_" + k: v for k, v in presentes.items()})
    return f, bruto, url

ficha_falha, html_falha, url_falha = ficha(linha_falha)
ficha_ok, html_ok, url_ok = ficha(linha_ok)

comparacao = pd.DataFrame([ficha_falha, ficha_ok]).T
comparacao.columns = ["não localizado", "extraído"]
print(comparacao.to_string())

os.makedirs(os.path.join(PASTA, "exemplo"), exist_ok=True)
comparacao.to_csv(os.path.join(PASTA, "exemplo", "comparacao.csv"))

## 5. Trecho do Item 8 nos dois documentos

O que costuma diferenciar os dois casos é o que vem escrito logo depois do título do item. Nos
documentos extraídos, seguem as demonstrações; nos não localizados, costuma haver uma remissão a
outro arquivo, a um anexo ausente ou a um documento incorporado por referência.

In [ ]:
def trecho_do_item(bruto, n=600):
    """Texto que segue o título do Item 8 (ou Item 18, no 20-F)."""
    texto, _ = texto_e_mapa(ENTIDADES.sub(" ", bruto))
    achados = list(P_10K_INI.finditer(texto)) or list(P_20F_INI.finditer(texto))
    if not achados:
        return "(título do item não encontrado no documento)"
    # o último costuma ser o do corpo, não o do sumário
    m = achados[-1]
    return texto[m.start():m.start() + n]

print("=" * 70)
print("NÃO LOCALIZADO —", ficha_falha["empresa"])
print("=" * 70)
print(trecho_do_item(html_falha))
print()
print("=" * 70)
print("EXTRAÍDO —", ficha_ok["empresa"])
print("=" * 70)
print(trecho_do_item(html_ok))

with open(os.path.join(PASTA, "exemplo", "trechos.txt"), "w", encoding="utf-8") as f:
    f.write("NAO LOCALIZADO - " + str(ficha_falha["empresa"]) + "\n")
    f.write(url_falha + "\n\n" + trecho_do_item(html_falha, 1200) + "\n\n\n")
    f.write("EXTRAIDO - " + str(ficha_ok["empresa"]) + "\n")
    f.write(url_ok + "\n\n" + trecho_do_item(html_ok, 1200) + "\n")
print("\ntrechos gravados em exemplo/trechos.txt")

## 6. Imagens para a figura

Converte a página em que aparece o título do item, em cada documento, para PNG. São essas duas
imagens que vão lado a lado na figura do artigo.

In [ ]:
!apt-get -qq install -y poppler-utils > /dev/null

def imagem_do_item(bruto, nome, paginas_max=3):
    """Renderiza o trecho ao redor do título do item e salva como PNG."""
    texto, mapa = texto_e_mapa(ENTIDADES.sub(" ", bruto))
    achados = list(P_10K_INI.finditer(texto)) or list(P_20F_INI.finditer(texto))
    html = ENTIDADES.sub(" ", bruto)
    if achados:
        ini = mapa[achados[-1].start()]
        fim = mapa[min(achados[-1].start() + 6000, len(mapa) - 1)]
        corpo = html[ini:fim]
    else:
        corpo = html[:40000]

    destino_pdf = os.path.join(PASTA, "exemplo", nome + ".pdf")
    gerar_pdf(corpo, nome, destino_pdf)

    saida = os.path.join(PASTA, "exemplo", nome)
    subprocess.run(["pdftoppm", "-png", "-r", "150", "-f", "1", "-l", str(paginas_max),
                    destino_pdf, saida], capture_output=True, timeout=300)
    geradas = sorted(glob.glob(saida + "*.png"))
    print(nome, "->", [os.path.basename(g) for g in geradas])
    return geradas

import glob
img_falha = imagem_do_item(html_falha, "A_nao_localizado")
img_ok = imagem_do_item(html_ok, "B_extraido")

In [ ]:
from IPython.display import Image, display

for caminho in (img_falha[:1] + img_ok[:1]):
    print(os.path.basename(caminho))
    display(Image(filename=caminho, width=700))

## O que dizer na legenda da figura

A diferença que a figura mostra não é de qualidade do documento, e sim de estrutura. Nos dois
casos o Item 8 existe e está corretamente identificado; no caso não localizado, o conteúdo das
demonstrações não está no documento examinado — foi apresentado em anexo separado, incorporado
por referência a outro arquivamento, ou disposto de forma que as âncoras usadas não alcançam.

A ficha da seção 4 dá os números para a legenda: quantos sinais contábeis cada documento
apresenta e quantos valores numéricos contém. O documento extraído costuma ter os seis sinais e
centenas ou milhares de valores; o não localizado apresenta parte dos sinais e conteúdo numérico
próximo de zero, porque o que ali está é remissão, não demonstração.